## Paired Comparison Methods for Ranking Leaderboard Data

### This notebook makes use of the `choix` library of paired choice ranking and analysis functions

In [1]:
pip install choix numpy pandas


[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

import pandas as pd
import json
import ast 

import numpy as np
import choix

### Motivation

In the generative AI ecosystem, leaderboards have become very popular for tracking the relative performance of LLMs and harnesses.
The purpose of creating a leaderboard is to assist people in making data driven choices from a variety of possible options.
However, there are some complexities with making use of leaderboard data. 

Consider the following scenario.
![Motivational Scenario](images/motivation.png)

As in the figure above, leaderboards often incorporate runs from multiple benchmarks.
The example above includes five benchmarks, A through E.
An advantage is that one can consider benchmark data from one or more use cases.
One disadvantage is that it introduces the problem of how to define a "global" ranking.
Taking the mean across benchmarks can provide an answer, but often the data do not include runs from all the same benchmarks.
You can see from the table above that no model has been run against the same set of benchmarks,
and so simply averaging scores must grapple with the fact that the average means something different for each model.
Does comparing means make sense when those means are taken over differing benchmark subsets?

One may also encounter situations where benchmark comparisons do not yield consistent ordering results.
In the figure above, benchmark B gives a different ordering of models X and Y than either benchmark A or C.
However, since X beats Y in 2 of 3 trials, we might wish to conclude that X beats Y, considering all the data we have available.

Lastly, we might wish to infer a conclusion about ordering of pairs which have no actual runs to compare them with.
For example, the above data do not provide any direct comparison between models X and Z,
but if we make an assumption of transitivity, we might wish to conclude that since model X is better than model Y,
and model Y is better than model Z, that model X is also better than model Z.
Bear in mind that this is an _assumption_, but it is a reasonable assumption based on the common property of transitivity.

One relevant conclusion from all of the above
is that it is best when we can pick one particular benchmark that represents our use case with good fidelity,
and simply use that benchmark to order our candidate models.
In some cases, however, we may not know _a priori_ what our use cases will be.
Or we might need to choose our models to cover a wide range of use cases.

In such situations, we may wish to identify a "best" model that takes into account all of the ambiguities described above.
For these situations, **paired comparison methods** are a good tool for imposing a single ordering
in the presence of these ambiguities, making use of the intuitive assumptions listed above. 

#### Let's run the example above with a paired choice ranking method from the `choix` library. We can see that it ranks the models according to the assumptions we described, in the presence of various ambiguous comparative runs:

In [3]:
# categories are the model names
cats = ["Model X", "Model Y", "Model Z"]

# (winner, loser) for each comparison
comps = [
    (0, 1),  # Model X beats Model Y  (benchmark A)
    (1, 0),  # Model Y beats Model X  (benchmark B)
    (0, 1),  # Model X beats Model Y  (benchmark C)
    (1, 2),  # Model Y beats Model Z  (benchmark D)
    (1, 2),  # Model Y beats Model Z  (benchmark E)
]

# run the pairwise comparison to get a ranking
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
   1. +2.627  Model X
   2. +1.938  Model Y
   3. -4.564  Model Z


#### The huggingface leaderboard I used for this notebook stores each benchmark run in its own JSON file. The following function reads all the run data in a directory and assembles it into a single pandas dataframe

In [4]:
def load_dicts_to_df(
    directory: str | Path,
    pattern: str = "*",
) -> pd.DataFrame:
    directory = Path(directory)
    if not directory.is_dir():
        raise NotADirectoryError(directory)

    rows: list[dict[str, Any]] = []
    # load any json files into the rows list
    for path in sorted(directory.glob(pattern)):
        if not path.is_file() or path.name.startswith("."):
            continue
        suffix = path.suffix.lower()
        data = None
        if suffix == ".json":
            with path.open(encoding="utf-8") as f:
                data = json.load(f)
        else:
            text = path.read_text(encoding="utf-8")
            data = ast.literal_eval(text)

        if not isinstance(data, dict):
            raise TypeError(f"{path} does not contain a dict (got {type(data).__name__})")
        rows.append(data)

    # the leaderboard json have a nested structure.
    # this function flattens the structure into flat column names
    return pd.json_normalize(rows)

#### Load the leaderboard data into a pandas dataframe

In [5]:
# git clone https://huggingface.co/spaces/taagarwa/coding-agent-leaderboard
df = load_dicts_to_df("/home/eje/git/coding-agent-leaderboard/results", pattern="*.json")
df.columns

Index(['benchmark.name', 'benchmark.repo', 'benchmark.num_tasks',
       'benchmark.url', 'harness.name', 'harness.skills', 'harness.is_oss',
       'harness.url', 'model.name', 'model.repo', 'model.is_oss',
       'model.num_params', 'model.precision', 'model.url', 'environment.name',
       'environment.config.name', 'environment.url', 'metrics.n_tasks',
       'metrics.n_errors', 'metrics.score', 'metrics.n_input_tokens',
       'metrics.n_cache_tokens', 'metrics.n_output_tokens',
       'metrics.n_total_tokens', 'metrics.agent_time_seconds',
       'metrics.total_time_seconds', 'metrics.cost_usd',
       'metrics.mean_input_tokens_per_task',
       'metrics.mean_cache_tokens_per_task',
       'metrics.mean_output_tokens_per_task', 'metrics.mean_tokens_per_task',
       'metrics.mean_cost_usd_per_task',
       'metrics.mean_total_time_seconds_per_task',
       'metrics.mean_agent_time_seconds_per_task', 'environment.config.path',
       'environment.config.version', 'environment.con

 #### The remainder of this notebook uses the following columns
 #### For the purposes of clarity, we will focus only on open models, and open harnesses, not proprietary

In [6]:
# Let's only consider open models
df = df.loc[df['model.is_oss'] == True]
# in fact let's only consider open harnesses too
df = df.loc[df['harness.is_oss'] == True]
df = df.reset_index(drop=True)
# keep only the columns we need
df = df[['model.name', 'harness.name', 'metrics.score', 'metrics.mean_cost_usd_per_task', 'metrics.mean_tokens_per_task', 'benchmark.name']]
df

,model.name,harness.name,metrics.score,metrics.mean_cost_usd_per_task,metrics.mean_tokens_per_task,benchmark.name
0,Gemma4-31B-FP8,OpenCode,0.417,0.20,1055516.0,SWE-Bench Pro -- Ansible
1,Gemma4-31B-FP8,Pi,0.469,0.17,833841.0,SWE-Bench Pro -- Ansible
2,Gemma4-31B-FP8,OpenCode,0.606,0.11,1059493.0,SWE-Bench Verified
3,Gemma4-31B-FP8,Pi,0.574,0.08,763314.0,SWE-Bench Verified
4,Nemotron-3-Super-120B-NVFP4,OpenCode,0.308,0.10,2367765.0,RH SWE-Bench
5,Nemotron-3-Super-120B-NVFP4,Pi,0.216,0.09,2176734.0,RH SWE-Bench
6,Nemotron-3-Super-120B-NVFP4,OpenCode,0.323,0.20,7180291.0,SWE-Bench Pro -- Ansible
7,Nemotron-3-Super-120B-NVFP4,Pi,0.375,0.27,10989983.0,SWE-Bench Pro -- Ansible
8,Nemotron-3-Super-120B-NVFP4,OpenCode,0.410,0.04,1461850.0,SWE-Bench Verified
9,Nemotron-3-Super-120B-NVFP4,Pi,0.498,0.07,2008071.0,SWE-Bench Verified


#### Remove some dirty data so the rankings behave better

In [7]:
df = df.dropna()
df = df.loc[df['metrics.mean_tokens_per_task'] > 0]
df = df.reset_index(drop=True)
df

,model.name,harness.name,metrics.score,metrics.mean_cost_usd_per_task,metrics.mean_tokens_per_task,benchmark.name
0,Gemma4-31B-FP8,OpenCode,0.417,0.20,1055516.0,SWE-Bench Pro -- Ansible
1,Gemma4-31B-FP8,Pi,0.469,0.17,833841.0,SWE-Bench Pro -- Ansible
2,Gemma4-31B-FP8,OpenCode,0.606,0.11,1059493.0,SWE-Bench Verified
3,Gemma4-31B-FP8,Pi,0.574,0.08,763314.0,SWE-Bench Verified
4,Nemotron-3-Super-120B-NVFP4,OpenCode,0.308,0.10,2367765.0,RH SWE-Bench
5,Nemotron-3-Super-120B-NVFP4,Pi,0.216,0.09,2176734.0,RH SWE-Bench
6,Nemotron-3-Super-120B-NVFP4,OpenCode,0.323,0.20,7180291.0,SWE-Bench Pro -- Ansible
7,Nemotron-3-Super-120B-NVFP4,Pi,0.375,0.27,10989983.0,SWE-Bench Pro -- Ansible
8,Nemotron-3-Super-120B-NVFP4,OpenCode,0.410,0.04,1461850.0,SWE-Bench Verified
9,Nemotron-3-Super-120B-NVFP4,Pi,0.498,0.07,2008071.0,SWE-Bench Verified


#### Our paired comparison method uses individual comparisons of the form (winner, loser). The following function can take a pandas table, and column names specifying which values are to be compared, and prepare the corresponding list of (winner, loser) pairs.

In [8]:
from typing import Any
from itertools import groupby
def prepare_ranking_data(df: pd.DataFrame,
                         catcol: str | list[str],
                         metcol: str,
                         descending: bool = False,
                         eqvcol: str | list[str] = []) -> tuple[list[tuple[int, int]], list[Any], list[float]]:
    ndata = df.shape[0]
    if ndata < 2:
        raise ValueError("Not enough data to prepare ranking comparisons")
    catcol = catcol if isinstance(catcol, list) else [catcol]
    eqvcol = eqvcol if isinstance(eqvcol, list) else [eqvcol]
    ncat = len(catcol)
    neqv = len(eqvcol)
    if ncat < 1:
        raise ValueError("No category column provided")
    tcols = catcol + eqvcol + [metcol]
    t = list(df[tcols].itertuples(index=False, name=None))
    metvals = [x[-1] for x in t]
    if ncat > 1:
        catvals = [x[:ncat] for x in t]
    else:
        # single category column: categories are just the values of the column
        catvals = [x[0] for x in t]
    if neqv > 0:
        # we will only compare items in the same equivalence group
        eqvvals = [x[ncat:ncat+neqv] for x in t]
    else:
        # by default all items are treated as one equivalence group
        eqvvals = [None] * ndata
    # map item categories to unique integers
    umap = dict([(y,x) for x,y in enumerate(sorted(set(catvals)))])
    compvals = [(umap[c],m,e) for c,m,e in zip(catvals, metvals, eqvvals)]
    comps = []
    for i in range(ndata):
        ic, im, ie = compvals[i]
        for j in range(i):
            jc, jm, je = compvals[j]
            if ie != je:
                # ignore items with different equivalence values
                continue
            if im == jm:
                # ignore items with same metric value
                continue
            # pairs are always of form (winner, loser)
            iwin = im < jm if descending else im > jm
            if iwin:
                comps.append((ic, jc))
            else:
                comps.append((jc, ic))
    return comps, sorted(umap.keys())

#### Rank our harnesses, using benchmark score as our metric. We will "integrate" over models, and only using comparisons where the benchmark is the same:

In [9]:
comps, cats = prepare_ranking_data(df, 'harness.name', 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")


Ranking:
   1. +0.418  Qwen Code
   2. +0.017  Pi
   3. -0.434  OpenCode


#### Rank our models using benchmark score, integrating over harness, and comparing within benchmark:

In [10]:
comps, cats = prepare_ranking_data(df, 'model.name', 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +3.851  Qwen3.6-27B-FP8
   2. +1.002  Qwen3.6-35B-A3B-NVFP4
   3. +0.715  Gemma4-31B-FP8
   4. -2.670  Nemotron-3-Super-120B-NVFP4
   5. -2.898  Mistral-Small-4-119B-2603-NVFP4


#### Rank (model, harness) pairs by benchmark score, comparing only within benchmark

In [11]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.score', eqvcol='benchmark.name')
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +8.251  ('Qwen3.6-27B-FP8', 'Pi')
   2. +7.139  ('Qwen3.6-27B-FP8', 'OpenCode')
   3. +6.582  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
   4. +2.352  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
   5. +1.628  ('Gemma4-31B-FP8', 'Pi')
   6. +0.900  ('Gemma4-31B-FP8', 'OpenCode')
   7. -3.815  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')
   8. -4.835  ('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')
   9. -5.091  ('Nemotron-3-Super-120B-NVFP4', 'Pi')
  10. -6.019  ('Nemotron-3-Super-120B-NVFP4', 'OpenCode')
  11. -7.092  ('Mistral-Small-4-119B-2603-NVFP4', 'Pi')


#### Rank (model, harness) pairs, by expected dollar cost per run. Here, we specify that lower cost is better:

In [12]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.mean_cost_usd_per_task', eqvcol='benchmark.name', descending=True)
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +7.746  ('Mistral-Small-4-119B-2603-NVFP4', 'Pi')
   2. +4.789  ('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')
   3. +0.814  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
   4. +0.511  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')
   5. -0.235  ('Nemotron-3-Super-120B-NVFP4', 'OpenCode')
   6. -0.870  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
   7. -1.388  ('Nemotron-3-Super-120B-NVFP4', 'Pi')
   8. -1.599  ('Gemma4-31B-FP8', 'Pi')
   9. -2.822  ('Qwen3.6-27B-FP8', 'OpenCode')
  10. -3.198  ('Gemma4-31B-FP8', 'OpenCode')
  11. -3.747  ('Qwen3.6-27B-FP8', 'Pi')


#### Rank (model, harness) pairs by expected token cost: here lower cost is better:

In [13]:
comps, cats = prepare_ranking_data(df, ['model.name', 'harness.name'], 'metrics.mean_tokens_per_task', eqvcol='benchmark.name', descending=True)
params = choix.ilsr_pairwise(len(cats), comps, alpha=1e-3)
ranking = np.argsort(params, descending=True)

print("Ranking:")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank:2d}. {params[idx]:+.3f}  {cats[idx]}")

Ranking:
   1. +4.810  ('Mistral-Small-4-119B-2603-NVFP4', 'Pi')
   2. +4.810  ('Gemma4-31B-FP8', 'Pi')
   3. +3.844  ('Mistral-Small-4-119B-2603-NVFP4', 'OpenCode')
   4. +2.962  ('Gemma4-31B-FP8', 'OpenCode')
   5. +1.706  ('Qwen3.6-27B-FP8', 'OpenCode')
   6. +1.664  ('Qwen3.6-35B-A3B-NVFP4', 'OpenCode')
   7. +0.729  ('Qwen3.6-35B-A3B-NVFP4', 'Qwen Code')
   8. -0.532  ('Qwen3.6-27B-FP8', 'Pi')
   9. -5.771  ('Nemotron-3-Super-120B-NVFP4', 'OpenCode')
  10. -6.661  ('Qwen3.6-35B-A3B-NVFP4', 'Pi')
  11. -7.561  ('Nemotron-3-Super-120B-NVFP4', 'Pi')
